# 01 — Dataset Exploration

This notebook explores the two benchmark datasets used throughout this project:

- **Split-CIFAR-100**: CIFAR-100 partitioned into 20 sequential tasks, each containing 5 disjoint classes.
- **Permuted-MNIST**: MNIST with a different fixed pixel permutation applied per task (10 tasks).

These are the standard benchmarks for class-incremental and domain-incremental continual learning respectively.

In [ ]:
import sys, os
sys.path.insert(0, os.path.join(os.getcwd(), '..'))

import numpy as np
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import torchvision
from torchvision import transforms
import torch

from src.preprocess import get_split_cifar100, get_permuted_mnist

%matplotlib inline
plt.rcParams.update({'figure.dpi': 120, 'font.size': 11})

## 1. Split-CIFAR-100

CIFAR-100 contains 100 classes with 500 training and 100 test images per class.
We shuffle the class order (seed=42) and divide into 20 non-overlapping tasks of 5 classes each.
Within each task, labels are remapped to 0–4 for a task-specific linear head.

In [ ]:
tasks_cifar = get_split_cifar100(n_tasks=20, batch_size=128, seed=42, data_root='../data')
print(f'Number of tasks : {len(tasks_cifar)}')
print(f'Classes per task: {tasks_cifar[0].n_classes}')
print()
for t in tasks_cifar[:5]:
    print(f'  Task {t.task_id:2d} | classes: {t.classes}')

In [ ]:
# CIFAR-100 class names
cifar100_classes = torchvision.datasets.CIFAR100(root='../data', train=False, download=True).classes

print('Task -> class names mapping (first 8 tasks):')
for t in tasks_cifar[:8]:
    names = [cifar100_classes[c] for c in t.classes]
    print(f'  Task {t.task_id:2d}: {names}')

In [ ]:
# Visualise sample images from tasks 0, 1, 2
denorm = transforms.Compose([
    transforms.Normalize(mean=[0,0,0], std=[1/0.2675, 1/0.2565, 1/0.2761]),
    transforms.Normalize(mean=[-0.5071, -0.4867, -0.4408], std=[1,1,1]),
])

fig, axes = plt.subplots(3, 5, figsize=(13, 8))
fig.suptitle('Sample images from Split-CIFAR-100\n(one row per task)', fontsize=13, fontweight='bold')

for row, task_idx in enumerate([0, 1, 2]):
    task = tasks_cifar[task_idx]
    loader = task.train_loader
    imgs, labels = next(iter(loader))
    # Pick one image per class (5 classes)
    shown = set()
    col = 0
    for img, lbl in zip(imgs, labels):
        if lbl.item() not in shown and col < 5:
            img_denorm = denorm(img).permute(1, 2, 0).clamp(0, 1).numpy()
            axes[row, col].imshow(img_denorm)
            class_name = cifar100_classes[task.classes[lbl.item()]]
            axes[row, col].set_title(f'T{task_idx} | {class_name}', fontsize=8)
            axes[row, col].axis('off')
            shown.add(lbl.item())
            col += 1

plt.tight_layout()
plt.savefig('../results/sample_cifar100_tasks.png', bbox_inches='tight')
plt.show()

In [ ]:
# Task sequence design: show how 100 classes map to 20 tasks
fig, ax = plt.subplots(figsize=(14, 4))

all_classes_by_task = [t.classes for t in tasks_cifar]

for t_idx, cls_list in enumerate(all_classes_by_task):
    for c in cls_list:
        ax.scatter(t_idx, c, s=25, color=f'C{t_idx % 10}', alpha=0.7)

ax.set_xlabel('Task index', fontsize=12)
ax.set_ylabel('CIFAR-100 class index', fontsize=12)
ax.set_title('Split-CIFAR-100 task partition (20 tasks × 5 classes)', fontsize=13)
ax.set_xticks(range(20))
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Permuted-MNIST

Each Permuted-MNIST task applies a fixed random permutation to the 784 pixel positions.
Task 0 uses the identity permutation (original MNIST).
All tasks share the same 10-class output (digit 0–9); only the input domain changes.
This is a domain-incremental continual learning scenario.

In [ ]:
tasks_mnist = get_permuted_mnist(n_tasks=10, batch_size=256, seed=42, data_root='../data')
print(f'Number of tasks  : {len(tasks_mnist)}')
print(f'Classes per task : {tasks_mnist[0].n_classes}')
print(f'Task names       : {[t.name for t in tasks_mnist]}')

In [ ]:
# Visualise the effect of different permutations on a single MNIST digit
base_imgs, base_lbl = next(iter(tasks_mnist[0].train_loader))
# Find a digit '5'
idx = (base_lbl == 5).nonzero(as_tuple=True)[0][0].item()

fig, axes = plt.subplots(2, 5, figsize=(13, 5))
fig.suptitle('Permuted-MNIST: same digit under different pixel permutations', fontsize=12, fontweight='bold')

for t_idx, task in enumerate(tasks_mnist):
    imgs, _ = next(iter(task.train_loader))
    img = imgs[idx].squeeze()
    row, col = t_idx // 5, t_idx % 5
    axes[row, col].imshow(img.numpy(), cmap='gray_r')
    axes[row, col].set_title(f'Task {t_idx}', fontsize=9)
    axes[row, col].axis('off')

plt.tight_layout()
plt.savefig('../results/sample_permuted_mnist.png', bbox_inches='tight')
plt.show()

## 3. Dataset Statistics

In [ ]:
print('='*55)
print(f'{"Dataset":<20} {"Tasks":>6} {"Classes/task":>13} {"Train/task":>12} {"Test/task":>11}')
print('-'*55)

t0c = tasks_cifar[0]
train_n = len(t0c.train_loader.dataset)
test_n  = len(t0c.test_loader.dataset)
print(f'{"Split-CIFAR-100":<20} {20:>6} {5:>13} {train_n:>12,} {test_n:>11,}')

t0m = tasks_mnist[0]
train_m = len(t0m.train_loader.dataset)
test_m  = len(t0m.test_loader.dataset)
print(f'{"Permuted-MNIST":<20} {10:>6} {10:>13} {train_m:>12,} {test_m:>11,}')
print('='*55)

## Summary

- **Split-CIFAR-100** (class-incremental): 20 tasks × 5 classes, ~2 500 train images / task.
- **Permuted-MNIST** (domain-incremental): 10 tasks × 10 classes, 60 000 train images / task.

In the following notebooks we train five continual learning strategies on these benchmarks
and compare them on average accuracy, backward transfer (forgetting), forward transfer,
and network sparsity.